In [1]:
import os
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np

# Paths
csv_path = "Data_Entry_2017_v2020.csv"
img_dir = "images"

# Load CSV
df = pd.read_csv(csv_path)

# Filter only images that exist in your current folder
available_imgs = set(os.listdir(img_dir))
df = df[df['Image Index'].isin(available_imgs)]

In [2]:
# Step 1: Define pneumonia-related findings
positive_findings = ['Pneumonia', 'Infiltration', 'Consolidation']

def is_pneumonia_related(label_str):
    labels = label_str.split('|')
    return any(l in positive_findings for l in labels)

# Step 2: Filter dataset to keep only pneumonia-related and no findings
def is_valid_entry(label_str):
    labels = label_str.split('|')
    return any(l in positive_findings + ['No Finding'] for l in labels)

df = df[df['Finding Labels'].apply(is_valid_entry)]
df['label'] = df['Finding Labels'].apply(lambda x: 1 if is_pneumonia_related(x) else 0)

# Step 3: Group patients and assign label = 1 if any image for that patient is positive
patient_df = df.groupby('Patient ID')['label'].max().reset_index()

# Step 4: Stratified split by patient label
train_patients, val_patients = train_test_split(
    patient_df,
    test_size=0.2,
    stratify=patient_df['label'],
    random_state=42
)

# Step 5: Split original DataFrame
train_df = df[df['Patient ID'].isin(train_patients['Patient ID'])]
val_df = df[df['Patient ID'].isin(val_patients['Patient ID'])]

# Print summary
print(f"Train images: {len(train_df)}, Val images: {len(val_df)}")
print(f"Train patients: {len(train_patients)}, Val patients: {len(val_patients)}\n")

print("Train class distribution:")
print(train_df['label'].value_counts())

print("\nValidation class distribution:")
print(val_df['label'].value_counts())

Train images: 3095, Val images: 677
Train patients: 948, Val patients: 238

Train class distribution:
label
0    2261
1     834
Name: count, dtype: int64

Validation class distribution:
label
0    493
1    184
Name: count, dtype: int64


In [3]:
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.loc[idx, 'Image Index']
        label = self.df.loc[idx, 'label']
        img_path = os.path.join(self.img_dir, img_name)

        image = Image.open(img_path).convert('L')  # grayscale
        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.float32)


In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])  # for grayscale
])

train_dataset = ChestXrayDataset(train_df, img_dir, transform)
val_dataset = ChestXrayDataset(val_df, img_dir, transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)


In [5]:
def imshow(img_tensor, title=None):
    # img_tensor shape: [1, H, W] because grayscale, so remove channel dim
    img = img_tensor.squeeze().numpy()  # remove channel dimension and convert to numpy
    img = img * 0.5 + 0.5  # unnormalize from [-1,1] to [0,1]
    plt.imshow(img, cmap='gray')
    if title:
        plt.title(title)
    plt.axis('off')
    plt.show()

In [6]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),  # grayscale input
            nn.ReLU(),
            nn.MaxPool2d(2),  # 112x112
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 56x56
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)   # 28x28
        )
        self.fc_block = nn.Sequential(
            nn.Linear(128 * 28 * 28, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 1),
            nn.Sigmoid()  # For binary output
        )

    def forward(self, x):
        x = self.conv_block(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.fc_block(x)
        return x


In [7]:
def train_one_epoch(loader, model, optimizer, criterion, device, epoch=None):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    loop = tqdm(loader, desc=f"Epoch {epoch+1} [Training]", leave=False)
    for images, labels in loop:
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)  # shape [batch, 1]

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # Logging
        running_loss += loss.item()
        preds = (outputs > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        loop.set_postfix(loss=loss.item(), accuracy=100 * correct / total)

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / total
    print(f"Epoch {epoch+1} - Train Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc*100:.2f}%")
    return epoch_loss, epoch_acc


In [8]:
def evaluate(loader, model, criterion, device, epoch=None):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        loop = tqdm(loader, desc=f"Epoch {epoch+1} [Validation]", leave=False)
        for images, labels in loop:
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            preds = (outputs > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            loop.set_postfix(loss=loss.item(), accuracy=100 * correct / total)

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / total
    print(f"Epoch {epoch+1} - Val Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc*100:.2f}%")
    return epoch_loss, epoch_acc


In [9]:
num_epochs = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleCNN().to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(train_loader, model, optimizer, criterion, device, epoch)
    val_loss, val_acc = evaluate(val_loader, model, criterion, device, epoch)


Epoch 1 - Train Loss: 0.5551 | Accuracy: 72.92%


Epoch 1 - Val Loss: 0.5469 | Accuracy: 73.12%


Epoch 2 - Train Loss: 0.5368 | Accuracy: 73.67%


Epoch 2 - Val Loss: 0.5511 | Accuracy: 73.12%


Epoch 3 - Train Loss: 0.5331 | Accuracy: 73.63%


...

Epoch 8 - Val Loss: 0.6022 | Accuracy: 73.41%


Epoch 9 - Train Loss: 0.4449 | Accuracy: 78.48%


Epoch 9 - Val Loss: 0.6062 | Accuracy: 72.67%


Epoch 10 - Train Loss: 0.4143 | Accuracy: 80.10%


Epoch 10 - Val Loss: 0.6629 | Accuracy: 72.97%
